# 01 · Exploración de Tablas Clave — BD Savia Salud EPS

**Objetivo:** Descargar las primeras 2 filas de cada tabla crítica del módulo de Cuentas Médicas (`cm_*`) y guardarlas como CSV en `data/database_analysis/` para entender la estructura real de los datos.

**Tablas exploradas:**
- `cm_facturas` — Cabecera de facturas
- `cm_detalles` — Líneas de servicios
- `cm_auditoria_motivos_glosas` — Motivos de glosa (TARGET del modelo)
- `cm_auditoria_devoluciones` — Devoluciones (TARGET clase 2)
- `cm_factura_estados` — Catálogo de estados
- `cm_fe_rips_cargas` — RIPS electrónicos (FEV)
- `cm_fe_rips_carga_contenidos` — JSON/XML almacenados
- `aseg_afiliados` — Datos del paciente
- `cnt_contratos` — Contratos con prestadores
- `cnt_prestadores` — Prestadores (IPS)
- y otras tablas auxiliares

**Salida:** `data/database_analysis/<tabla>_muestra.csv` por cada tabla

In [1]:
# Celda 1 — Conexión a la base de datos
# Configuración idéntica a extraccion_xml_json.ipynb
import os
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# --- Credenciales de conexión ---
user     = "jcardonr"
password = "6nxx2yVWaKXyiF1cZbfS*"
host     = "10.250.5.36"
db_name  = "system_savia"
port     = "3306"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}",
    connect_args={"connect_timeout": 30}
)

# Carpeta de salida
OUTPUT_DIR = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo\data\database_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verificar conexión
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Conexión exitosa → {host}/{db_name}")
    print(f"📁 Carpeta de salida: {OUTPUT_DIR}")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

✅ Conexión exitosa → 10.250.5.36/system_savia
📁 Carpeta de salida: D:\Users\jcardonr\Documents\eafit\Codigo\data\database_analysis


In [2]:
# Celda 2 — Lista de tablas a explorar (desde queries_exploracion_cm.sql)

TABLAS = [
    # Tablas críticas para el modelo (Tier 1)
    "cm_facturas",
    "cm_detalles",
    "cm_auditoria_motivos_glosas",
    "cm_auditoria_devoluciones",
    "cm_factura_estados",
    "cm_fe_rips_cargas",
    "cm_fe_rips_carga_contenidos",
    "aseg_afiliados",
    "cnt_contratos",
    "cnt_prestadores",
    # Tablas de soporte (Tier 2)
    "aa_mala_cm_fe_rips_cargas",
    "aa_temp_cm_fe_rips_cargas",
    "aseg_afiliados_ods",
    "v_cnt_prestadores_axcel",
    "ws_cm_facturas",
]

print(f"Tablas a explorar: {len(TABLAS)}")
for t in TABLAS:
    print(f"  - {t}")

Tablas a explorar: 15
  - cm_facturas
  - cm_detalles
  - cm_auditoria_motivos_glosas
  - cm_auditoria_devoluciones
  - cm_factura_estados
  - cm_fe_rips_cargas
  - cm_fe_rips_carga_contenidos
  - aseg_afiliados
  - cnt_contratos
  - cnt_prestadores
  - aa_mala_cm_fe_rips_cargas
  - aa_temp_cm_fe_rips_cargas
  - aseg_afiliados_ods
  - v_cnt_prestadores_axcel
  - ws_cm_facturas


In [3]:
# Celda 3 — Descarga de 2 filas por tabla y almacenamiento en CSV

resultados = []  # Registro de éxitos/errores

for tabla in TABLAS:
    try:
        df = pd.read_sql(f"SELECT * FROM {tabla} LIMIT 2", con=engine)
        
        # Guardar CSV
        archivo = OUTPUT_DIR / f"{tabla}_muestra.csv"
        df.to_csv(archivo, index=False, encoding="utf-8-sig")
        
        resultados.append({
            "tabla": tabla,
            "filas": len(df),
            "columnas": len(df.columns),
            "estado": "OK",
            "archivo": archivo.name,
        })
        print(f"✅ {tabla:45s} → {len(df.columns):3d} cols | {archivo.name}")
        
    except Exception as e:
        resultados.append({
            "tabla": tabla,
            "filas": 0,
            "columnas": 0,
            "estado": f"ERROR: {e}",
            "archivo": None,
        })
        print(f"❌ {tabla:45s} → {str(e)[:60]}")

df_resultados = pd.DataFrame(resultados)
print(f"\nResumen: {df_resultados[df_resultados['estado']=='OK'].shape[0]} tablas OK | "
      f"{df_resultados[df_resultados['estado']!='OK'].shape[0]} errores")

✅ cm_facturas                                   →  68 cols | cm_facturas_muestra.csv
✅ cm_detalles                                   →  63 cols | cm_detalles_muestra.csv
✅ cm_auditoria_motivos_glosas                   →  19 cols | cm_auditoria_motivos_glosas_muestra.csv
✅ cm_auditoria_devoluciones                     →  30 cols | cm_auditoria_devoluciones_muestra.csv
✅ cm_factura_estados                            →   7 cols | cm_factura_estados_muestra.csv
✅ cm_fe_rips_cargas                             →  63 cols | cm_fe_rips_cargas_muestra.csv
✅ cm_fe_rips_carga_contenidos                   →  10 cols | cm_fe_rips_carga_contenidos_muestra.csv
✅ aseg_afiliados                                → 200 cols | aseg_afiliados_muestra.csv
✅ cnt_contratos                                 →  32 cols | cnt_contratos_muestra.csv
✅ cnt_prestadores                               →  30 cols | cnt_prestadores_muestra.csv
✅ aa_mala_cm_fe_rips_cargas                     →  63 cols | aa_mala_cm_fe_rips_ca

In [4]:
# Celda 4 — Inspección visual de cada tabla descargada

for tabla in TABLAS:
    archivo = OUTPUT_DIR / f"{tabla}_muestra.csv"
    if not archivo.exists():
        print(f"\n⚠️  {tabla} — no descargada (error)")
        continue
    
    df = pd.read_csv(archivo)
    
    print(f"\n{'='*70}")
    print(f"📋 {tabla} ({len(df.columns)} columnas)")
    print(f"{'='*70}")
    print("Columnas:", df.columns.tolist())
    print()
    display(df)


📋 cm_facturas (68 columnas)
Columnas: ['id', 'gn_empresas_id', 'cm_grupos_id', 'cnt_prestadores_id', 'cm_rips_cargas_id', 'gn_usuarios_lider_id', 'gn_usuarios_tecnico_id', 'gn_usuarios_medico_id', 'gn_usuarios_gestiona_id', 'cm_fe_rips_cargas_id', 'mae_tipo_contrato_id', 'mae_tipo_contrato_codigo', 'mae_tipo_contrato_valor', 'nit', 'ips', 'numero_radicado', 'numero_facturado', 'fecha_prestacion', 'fecha_radicacion', 'multiusuario', 'valor_pendiente_actual', 'valor_inicial_glosa', 'marcacion', 'fecha_marcacion', 'fecha_vencimiento', 'tipo_auditoria', 'historial_proceso', 'valor_factura', 'valor_pagado_factura', 'valor_copago', 'valor_bruto', 'valor_cuota_moderadora', 'mae_regimen_id', 'mae_regimen_codigo', 'mae_regimen_valor', 'estado_factura', 'pbs', 'numero_contrato', 'numero_contrato_auditoria', 'impuesto_timbre', 'origen_factura', 'usuario_audita', 'terminal_audita', 'fecha_hora_audita', 'usuario_devuelve', 'terminal_devuelve', 'fecha_hora_devuelve', 'usuario_respuesta', 'terminal_

,id,gn_empresas_id,cm_grupos_id,cnt_prestadores_id,cm_rips_cargas_id,gn_usuarios_lider_id,gn_usuarios_tecnico_id,gn_usuarios_medico_id,gn_usuarios_gestiona_id,cm_fe_rips_cargas_id,...,ejecucion_glosa_aut_bot,valor_no_acuerdo,ejecucion_glosa_aut_ia,estado_glosa_aut,usuario_crea,terminal_crea,fecha_hora_crea,usuario_modifica,terminal_modifica,fecha_hora_modifica
0,1,1,4,517,345294,1368,NaN,NaN,1368,NaN,...,0,NaN,0,0,ygomezda,10.250.0.129,2022-09-05 18:11:08,NaN,NaN,NaN
1,4,1,4,364,63,2255,2266.0,NaN,2266,NaN,...,0,NaN,0,0,mcarmona,127.0.0.1,2021-01-05 11:43:35,YEIRA YUBIANA SUAREZ PATERNINA,10.250.0.129,2021-04-18 17:04:33



📋 cm_detalles (63 columnas)
Columnas: ['id', 'cm_facturas_id', 'aseg_afiliados_id', 'estado', 'mae_tipo_documento_id', 'mae_tipo_documento_codigo', 'mae_tipo_documento_valor', 'documento', 'nombre_completo_afiliado', 'consecutivo_item', 'ma_servicio_id', 'ma_servicio_codigo', 'ma_servicio_valor', 'cantidad', 'radicado_glosa', 'valor_copago', 'valor_facturado', 'valor_unitario', 'valor_pagado', 'valor_pendiente', 'valor_pendiente_actual', 'valor_aceptado_ips', 'observacion', 'valor_pagado_eps', 'porcentaje_pagado_eps', 'porcentaje_aceptado_ips', 'observacion_respuesta_detalles', 'aplica_ac', 'aplica_dc', 'aplica_pbs', 'tipo_servicio', 'fecha_prestacion', 'valor_copago_item', 'aplica_glosa', 'aplica_concepto', 'aplica_dx', 'aplica_autorizacion', 'valor_glosa', 'motivo_glosa', 'concepto_contable', 'porcentaje_concepto', 'numero_autorizacion', 'codigo_dx', 'nombre_dx', 'mae_ambito_id', 'mae_ambito_codigo', 'mae_ambito_valor', 'copago_no_efectivo', 'aplica_recobro', 'mipres_numero_prescrip

,id,cm_facturas_id,aseg_afiliados_id,estado,mae_tipo_documento_id,mae_tipo_documento_codigo,mae_tipo_documento_valor,documento,nombre_completo_afiliado,consecutivo_item,...,mae_motivo_respuesta_glosa_especifico_valor,mae_motivo_respuesta_glosa_aplicacion_id,mae_motivo_respuesta_glosa_aplicacion_codigo,mae_motivo_respuesta_glosa_aplicacion_valor,usuario_crea,terminal_crea,fecha_hora_crea,usuario_modifica,terminal_modifica,fecha_hora_modifica
0,216,4,2313931,0,401,CC,Cedula Ciudadania,1045142585,SEBASTIAN DE JESUS AGUIRRE MARIN,1,...,NaN,NaN,NaN,NaN,mcarmona,127.0.0.1,2021-01-05 11:43:35,NaN,NaN,NaN
1,217,4,2313931,0,401,CC,Cedula Ciudadania,1045142585,SEBASTIAN DE JESUS AGUIRRE MARIN,2,...,NaN,NaN,NaN,NaN,mcarmona,127.0.0.1,2021-01-05 11:43:35,NaN,NaN,NaN



📋 cm_auditoria_motivos_glosas (19 columnas)
Columnas: ['id', 'cm_detalles_id', 'mae_motivo_id', 'mae_motivo_codigo', 'mae_motivo_valor', 'mae_motivo_especifico_id', 'mae_motivo_especifico_codigo', 'mae_motivo_especifico_valor', 'mae_motivo_glosa_aplicacion_id', 'mae_motivo_glosa_aplicacion_codigo', 'mae_motivo_glosa_aplicacion_valor', 'mae_motivo_glosa_aplicacion_descripcion', 'porcentaje', 'valor_motivo', 'observacion', 'tipologia', 'usuario_crea', 'terminal_crea', 'fecha_hora_crea']



,id,cm_detalles_id,mae_motivo_id,mae_motivo_codigo,mae_motivo_valor,mae_motivo_especifico_id,mae_motivo_especifico_codigo,mae_motivo_especifico_valor,mae_motivo_glosa_aplicacion_id,mae_motivo_glosa_aplicacion_codigo,mae_motivo_glosa_aplicacion_valor,mae_motivo_glosa_aplicacion_descripcion,porcentaje,valor_motivo,observacion,tipologia,usuario_crea,terminal_crea,fecha_hora_crea
0,2,210164,9601,2,Tarifas,9639,202,"02 Consultas, interconsultas y visitas médicas",61152,RE3047,No Aplica,No Aplica,2.4,1996.0,se objeta mayor valor facturado según tarifa ...,0,jmarines (JOHN FERNANDO MARIN ESPINOSA),10.250.0.129,2021-01-18 16:05:52
1,3,146849,9601,2,Tarifas,9610,102,"Consultas, interconsultas y visitas médicas",61152,RE3047,No Aplica,No Aplica,10.0,2660.0,glosa,0,ysuarezp (YEIRA YUBIANA SUAREZ PATERNINA),10.250.0.129,2021-01-18 16:25:05


... (20 outputs left out)


📋 v_cnt_prestadores_axcel (6 columnas)
Columnas: ['numero_documento', 'razon_social', 'codigo_habilitacion', 'ubicacion', 'direccion', 'nombre_sede']



,numero_documento,razon_social,codigo_habilitacion,ubicacion,direccion,nombre_sede
0,15323342,CARLOS ARTURO TORRES GOMEZ,1532334201,YARUMAL,SIN DIRECCION,CARLOS ARTURO TORRES GOMEZ
1,890980068,LA CASA DEL BUEN DIOS,890980068001,MEDELLÍN,Carrera 50 # 63 - 136,LA CASA DEL BUEN DIOS



📋 ws_cm_facturas (21 columnas)
Columnas: ['id', 'cm_radicados_id', 'cm_facturas_id', 'estado', 'proveedor_nit', 'numero_documento', 'numero_radicado', 'factura_id', 'regimen', 'valor_documento', 'valor_pagado', 'valor_glosado', 'valor_copago', 'fecha_hora_documento', 'fecha_hora_proceso', 'cuota_moderadora', 'contrato', 'impuesto_timbre', 'usuario_crea', 'terminal_crea', 'fecha_hora_crea']



,id,cm_radicados_id,cm_facturas_id,estado,proveedor_nit,numero_documento,numero_radicado,factura_id,regimen,valor_documento,...,valor_glosado,valor_copago,fecha_hora_documento,fecha_hora_proceso,cuota_moderadora,contrato,impuesto_timbre,usuario_crea,terminal_crea,fecha_hora_crea
0,2100,5123247,17450847,3,901249947,FE76032,460641,17450847,SUBSIDIADO,139370.0,...,5435.4300,0.0,2023-02-23,2023-03-13 15:04:25,NaN,0229-2021,b'\x00',pgomezva (PAULA YECENIA GOMEZ VALENCIA),10.250.0.129,2023-03-23 08:59:28
1,2101,5123246,17450566,3,901249947,FE74739,460623,17450566,SUBSIDIADO,64048.0,...,5065.7376,7100.0,2023-02-08,2023-03-13 14:56:04,NaN,0229-2021,b'\x00',jgonzalm (JAVIER EMILIO GONZALEZ MONSALVE),10.250.0.129,2023-03-23 08:59:32


In [5]:
# Celda 5 — Conteo real de registros en cada tabla
# Importante para entender el volumen disponible para entrenamiento

conteos = []
tablas_conteo = [
    "cm_facturas",
    "cm_detalles",
    "cm_auditoria_motivos_glosas",
    "cm_auditoria_devoluciones",
    "cm_fe_rips_cargas",
    "cm_fe_rips_carga_contenidos",
    "aseg_afiliados",
    "cnt_contratos",
    "cnt_prestadores",
]

print("--- Conteo total de registros por tabla ---")
for tabla in tablas_conteo:
    try:
        res = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", con=engine)
        total = res["total"][0]
        conteos.append({"tabla": tabla, "total_registros": total})
        print(f"  {tabla:45s} → {total:>12,} registros")
    except Exception as e:
        print(f"  {tabla:45s} → ERROR: {e}")

df_conteos = pd.DataFrame(conteos)
df_conteos.to_csv(OUTPUT_DIR / "00_conteo_registros.csv", index=False)
print(f"\n✅ Guardado: 00_conteo_registros.csv")

--- Conteo total de registros por tabla ---
  cm_facturas                                   →   12,471,822 registros
  cm_detalles                                   →  260,129,137 registros
  cm_auditoria_motivos_glosas                   →   71,778,926 registros
  cm_auditoria_devoluciones                     →      550,837 registros
  cm_fe_rips_cargas                             →    2,051,167 registros
  cm_fe_rips_carga_contenidos                   →    6,152,045 registros
  aseg_afiliados                                →    3,603,086 registros
  cnt_contratos                                 →        2,915 registros
  cnt_prestadores                               →       53,563 registros

✅ Guardado: 00_conteo_registros.csv


In [6]:
# Celda 6 — Análisis específico: columnas de cm_facturas relacionadas con el TARGET
# Buscar campos de estado, glosa y devolución

archivo_facturas = OUTPUT_DIR / "cm_facturas_muestra.csv"
if archivo_facturas.exists():
    df_f = pd.read_csv(archivo_facturas)
    
    print("COLUMNAS DE cm_facturas:")
    print(f"  Total: {len(df_f.columns)} columnas")
    print()
    
    # Clasificar columnas por categoría
    cols = df_f.columns.tolist()
    
    cat_estado  = [c for c in cols if any(p in c.lower() for p in ["estado","estado_"])]
    cat_glosa   = [c for c in cols if "glosa" in c.lower()]
    cat_valor   = [c for c in cols if "valor" in c.lower()]
    cat_fecha   = [c for c in cols if "fecha" in c.lower()]
    cat_fk      = [c for c in cols if c.endswith("_id")]
    cat_otros   = [c for c in cols if c not in cat_estado+cat_glosa+cat_valor+cat_fecha+cat_fk]
    
    print(f"  🎯 Columnas de ESTADO (candidatas TARGET): {cat_estado}")
    print(f"  💰 Columnas de GLOSA:                      {cat_glosa}")
    print(f"  💵 Columnas de VALOR:                      {cat_valor}")
    print(f"  📅 Columnas de FECHA:                      {cat_fecha}")
    print(f"  🔗 Columnas FK (_id):                      {cat_fk}")
    print(f"  📝 Otras columnas:                         {cat_otros}")
    print()
    print("Valores de ejemplo (primeras 2 filas):")
    display(df_f)

COLUMNAS DE cm_facturas:
  Total: 68 columnas

  🎯 Columnas de ESTADO (candidatas TARGET): ['cnt_prestadores_id', 'estado_factura', 'estado_glosa_aut']
  💰 Columnas de GLOSA:                      ['valor_inicial_glosa', 'ejecucion_glosa_aut_bot', 'ejecucion_glosa_aut_ia', 'estado_glosa_aut']
  💵 Columnas de VALOR:                      ['mae_tipo_contrato_valor', 'valor_pendiente_actual', 'valor_inicial_glosa', 'valor_factura', 'valor_pagado_factura', 'valor_copago', 'valor_bruto', 'valor_cuota_moderadora', 'mae_regimen_valor', 'valor_no_acuerdo']
  📅 Columnas de FECHA:                      ['fecha_prestacion', 'fecha_radicacion', 'fecha_marcacion', 'fecha_vencimiento', 'fecha_hora_audita', 'fecha_hora_devuelve', 'fecha_hora_respuesta', 'fecha_hora_concilia', 'fecha_marcacion_respuesta_ips', 'fecha_asignacion_tecnico', 'fecha_asignacion_medico', 'fecha_hora_crea', 'fecha_hora_modifica']
  🔗 Columnas FK (_id):                      ['gn_empresas_id', 'cm_grupos_id', 'cnt_prestadores_id', 

,id,gn_empresas_id,cm_grupos_id,cnt_prestadores_id,cm_rips_cargas_id,gn_usuarios_lider_id,gn_usuarios_tecnico_id,gn_usuarios_medico_id,gn_usuarios_gestiona_id,cm_fe_rips_cargas_id,...,ejecucion_glosa_aut_bot,valor_no_acuerdo,ejecucion_glosa_aut_ia,estado_glosa_aut,usuario_crea,terminal_crea,fecha_hora_crea,usuario_modifica,terminal_modifica,fecha_hora_modifica
0,1,1,4,517,345294,1368,NaN,NaN,1368,NaN,...,0,NaN,0,0,ygomezda,10.250.0.129,2022-09-05 18:11:08,NaN,NaN,NaN
1,4,1,4,364,63,2255,2266.0,NaN,2266,NaN,...,0,NaN,0,0,mcarmona,127.0.0.1,2021-01-05 11:43:35,YEIRA YUBIANA SUAREZ PATERNINA,10.250.0.129,2021-04-18 17:04:33


In [7]:
# Celda 7 — Análisis: ¿cm_auditoria_motivos_glosas tiene el TARGET?
# Esta tabla contiene los motivos de glosa → es la fuente del TARGET clase 1

archivo_glosas = OUTPUT_DIR / "cm_auditoria_motivos_glosas_muestra.csv"
if archivo_glosas.exists():
    df_g = pd.read_csv(archivo_glosas)
    print("COLUMNAS DE cm_auditoria_motivos_glosas:")
    print(f"  Total: {len(df_g.columns)} columnas")
    print(f"  Columnas: {df_g.columns.tolist()}")
    print()
    display(df_g)
    
    # Verificar FK hacia cm_detalles
    fk_cols = [c for c in df_g.columns if c.endswith("_id")]
    print(f"\n  🔗 FKs detectadas: {fk_cols}")
    print("  → La FK 'cm_detalles_id' vincula la glosa con la línea de servicio")
else:
    print("⚠️  Tabla no descargada")

COLUMNAS DE cm_auditoria_motivos_glosas:
  Total: 19 columnas
  Columnas: ['id', 'cm_detalles_id', 'mae_motivo_id', 'mae_motivo_codigo', 'mae_motivo_valor', 'mae_motivo_especifico_id', 'mae_motivo_especifico_codigo', 'mae_motivo_especifico_valor', 'mae_motivo_glosa_aplicacion_id', 'mae_motivo_glosa_aplicacion_codigo', 'mae_motivo_glosa_aplicacion_valor', 'mae_motivo_glosa_aplicacion_descripcion', 'porcentaje', 'valor_motivo', 'observacion', 'tipologia', 'usuario_crea', 'terminal_crea', 'fecha_hora_crea']



,id,cm_detalles_id,mae_motivo_id,mae_motivo_codigo,mae_motivo_valor,mae_motivo_especifico_id,mae_motivo_especifico_codigo,mae_motivo_especifico_valor,mae_motivo_glosa_aplicacion_id,mae_motivo_glosa_aplicacion_codigo,mae_motivo_glosa_aplicacion_valor,mae_motivo_glosa_aplicacion_descripcion,porcentaje,valor_motivo,observacion,tipologia,usuario_crea,terminal_crea,fecha_hora_crea
0,2,210164,9601,2,Tarifas,9639,202,"02 Consultas, interconsultas y visitas médicas",61152,RE3047,No Aplica,No Aplica,2.4,1996.0,se objeta mayor valor facturado según tarifa ...,0,jmarines (JOHN FERNANDO MARIN ESPINOSA),10.250.0.129,2021-01-18 16:05:52
1,3,146849,9601,2,Tarifas,9610,102,"Consultas, interconsultas y visitas médicas",61152,RE3047,No Aplica,No Aplica,10.0,2660.0,glosa,0,ysuarezp (YEIRA YUBIANA SUAREZ PATERNINA),10.250.0.129,2021-01-18 16:25:05



  🔗 FKs detectadas: ['cm_detalles_id', 'mae_motivo_id', 'mae_motivo_especifico_id', 'mae_motivo_glosa_aplicacion_id']
  → La FK 'cm_detalles_id' vincula la glosa con la línea de servicio


In [8]:
# Celda 8 — Análisis: cm_auditoria_devoluciones (TARGET clase 2)

archivo_dev = OUTPUT_DIR / "cm_auditoria_devoluciones_muestra.csv"
if archivo_dev.exists():
    df_d = pd.read_csv(archivo_dev)
    print("COLUMNAS DE cm_auditoria_devoluciones:")
    print(f"  Total: {len(df_d.columns)} columnas")
    print(f"  Columnas: {df_d.columns.tolist()}")
    print()
    display(df_d)
else:
    print("⚠️  Tabla no descargada — Verificar que exista con ese nombre")
    
    # Buscar tablas de devolución alternativas
    print("\nBuscando tablas de devolución en la BD...")
    try:
        res = pd.read_sql(
            "SELECT table_name FROM information_schema.tables "
            "WHERE table_schema = 'system_savia' AND table_name LIKE '%devol%'",
            con=engine
        )
        print(f"  Tablas con 'devol': {res['table_name'].tolist()}")
    except Exception as e:
        print(f"  Error buscando tablas: {e}")

COLUMNAS DE cm_auditoria_devoluciones:
  Total: 30 columnas
  Columnas: ['id', 'cm_facturas_id', 'cm_devolucion_masiva_id', 'mae_motivo_devolucion_id', 'mae_motivo_devolucion_codigo', 'mae_motivo_devolucion_valor', 'mae_contrato_modalidad_id', 'mae_contrato_modalidad_codigo', 'mae_contrato_modalidad_valor', 'mae_regimen_id', 'mae_regimen_codigo', 'mae_regimen_valor', 'nit', 'ips', 'numero_radicado', 'numero_facturado', 'fecha_radicacion', 'fecha_devolucion', 'valor_factura', 'observacion', 'mae_devolucion_motivo_general_id', 'mae_devolucion_motivo_general_codigo', 'mae_devolucion_motivo_general_valor', 'mae_devolucion_motivo_general_descripcion', 'usuario_crea', 'terminal_crea', 'fecha_hora_crea', 'usuario_modifica', 'terminal_modifica', 'fecha_hora_modifica']



,id,cm_facturas_id,cm_devolucion_masiva_id,mae_motivo_devolucion_id,mae_motivo_devolucion_codigo,mae_motivo_devolucion_valor,mae_contrato_modalidad_id,mae_contrato_modalidad_codigo,mae_contrato_modalidad_valor,mae_regimen_id,...,mae_devolucion_motivo_general_id,mae_devolucion_motivo_general_codigo,mae_devolucion_motivo_general_valor,mae_devolucion_motivo_general_descripcion,usuario_crea,terminal_crea,fecha_hora_crea,usuario_modifica,terminal_modifica,fecha_hora_modifica
0,1,18878,NaN,9726,849,Factura no cumple requisitos legales,9118,2,Evento,9304,...,60755,3047,Resolución 3047,NaN,jsepulvc (JOHANA MARCELA SEPULVEDA CORREA),10.250.0.129,2021-01-20 15:05:20,NaN,NaN,NaN
1,2,18761,NaN,9726,849,Factura no cumple requisitos legales,9118,2,Evento,9304,...,60755,3047,Resolución 3047,NaN,agarciaa (ANA ISABEL GARCIA ACEVEDO),10.250.0.129,2021-01-20 16:23:57,NaN,NaN,NaN


In [9]:
# Celda 9 — Análisis: cm_fe_rips_carga_contenidos (JSON y XML)
# Esta tabla contiene el JSON RIPS y XML FEV — datos clínicos detallados

archivo_contenidos = OUTPUT_DIR / "cm_fe_rips_carga_contenidos_muestra.csv"
if archivo_contenidos.exists():
    df_c = pd.read_csv(archivo_contenidos)
    print("COLUMNAS DE cm_fe_rips_carga_contenidos:")
    print(f"  Total: {len(df_c.columns)} columnas")
    print(f"  Columnas: {df_c.columns.tolist()}")
    print()
    
    # Mostrar primeras filas ocultando el contenido JSON/XML (puede ser muy largo)
    df_preview = df_c.copy()
    for col in ["json", "xml", "cuv_json"]:
        if col in df_preview.columns:
            df_preview[col] = df_preview[col].astype(str).str[:80] + "..."
    display(df_preview)
    
    # Mostrar un ejemplo del JSON RIPS si existe
    if "json" in df_c.columns and pd.notna(df_c["json"].iloc[0]):
        import json
        print("\nEstructura del JSON RIPS (primeros 500 chars):")
        try:
            json_sample = str(df_c["json"].iloc[0])[:500]
            print(json_sample)
        except Exception:
            print("  No se pudo parsear el JSON")
else:
    print("⚠️  Tabla no descargada")

COLUMNAS DE cm_fe_rips_carga_contenidos:
  Total: 10 columnas
  Columnas: ['id', 'cm_fe_rips_cargas_id', 'tipo', 'json', 'xml', 'cuv', 'cuv_json', 'usuario_crea', 'terminal_crea', 'fecha_hora_crea']



,id,cm_fe_rips_cargas_id,tipo,json,xml,cuv,cuv_json,usuario_crea,terminal_crea,fecha_hora_crea
0,1,2000000,1,"{""numNota"": null, ""tipoNota"": null, ""usuarios""...",nan...,NaN,nan...,900226451r01 (ESPECIALIDADES MEDICAS METROPOLI...,192.168.255.17,2025-02-14 10:18:31
1,2,2000000,2,nan...,nan...,Proceso Id: 208\nFactura: 1163809\nFecha: mart...,nan...,900226451r01 (ESPECIALIDADES MEDICAS METROPOLI...,192.168.255.17,2025-02-14 10:18:31



Estructura del JSON RIPS (primeros 500 chars):
{"numNota": null, "tipoNota": null, "usuarios": [{"codSexo": "F", "servicios": {"procedimientos": [{"idMIPRES": null, "vrServicio": 35326, "codServicio": 745, "consecutivo": 1, "codPrestador": "050881081901", "grupoServicios": "02", "codComplicacion": null, "conceptoRecaudo": "05", "numAutorizacion": null, "codProcedimiento": "881201", "valorPagoModerador": 0, "fechaInicioAtencion": "2025-02-03 15:23", "numFEVPagoModerador": null, "codDiagnosticoPrincipal": "R688", "viaIngresoServicioSalud": "01


In [10]:
# Celda 10 — Verificar join entre cm_facturas y cm_auditoria_motivos_glosas
# Confirmar que la ruta: cm_facturas → cm_detalles → cm_auditoria_motivos_glosas funciona

query_join_test = """
SELECT
    f.id                        AS factura_id,
    f.valor_factura,
    COUNT(DISTINCT d.id)        AS num_detalles,
    COUNT(camg.id)              AS num_glosas,
    SUM(camg.valor_motivo)      AS valor_total_glosa,
    CASE
        WHEN COUNT(camg.id) > 0 THEN 1
        ELSE 0
    END                         AS es_glosada
FROM cm_facturas f
LEFT JOIN cm_detalles d
    ON d.cm_facturas_id = f.id
LEFT JOIN cm_auditoria_motivos_glosas camg
    ON camg.cm_detalles_id = d.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 3 MONTH)
GROUP BY f.id, f.valor_factura
ORDER BY num_glosas DESC
LIMIT 20;
"""

try:
    df_join = pd.read_sql(query_join_test, con=engine)
    print("✅ Join cm_facturas → cm_detalles → cm_auditoria_motivos_glosas: FUNCIONA")
    print(f"\nFacturas con glosas detectadas: {df_join[df_join['es_glosada']==1].shape[0]}")
    print(f"Facturas sin glosas: {df_join[df_join['es_glosada']==0].shape[0]}")
    display(df_join)
    df_join.to_csv(OUTPUT_DIR / "99_test_join_glosas.csv", index=False)
except Exception as e:
    print(f"❌ Error en join: {e}")
    print("   → Verificar nombres de columna FK en cm_detalles (puede ser 'facturas_id' en lugar de 'cm_facturas_id')")

✅ Join cm_facturas → cm_detalles → cm_auditoria_motivos_glosas: FUNCIONA

Facturas con glosas detectadas: 20
Facturas sin glosas: 0


,factura_id,valor_factura,num_detalles,num_glosas,valor_total_glosa,es_glosada
0,23844966,899342793.0,3158,6489,1.466253e+09,1
1,23757825,75958472.0,2184,4408,1.575589e+08,1
2,23874233,193561199.0,2026,4085,4.074776e+08,1
3,23847626,213764108.0,2008,4023,4.308478e+08,1
4,23751741,156110155.0,1701,3530,3.200964e+08,1
5,23873840,265808432.0,1730,3517,5.959455e+08,1
6,23700130,335293596.0,1648,3294,6.704122e+08,1
7,23779055,114992768.0,1561,3249,2.680139e+08,1
8,23838643,178417020.0,1459,3188,4.037019e+08,1
9,23844802,206232890.0,1358,2728,4.197979e+08,1


In [ ]:
# Celda 11 — Distribución real de estados: Auditada / Glosada / Devuelta
# CORRECCIÓN (2026-03-04): cm_auditoria_devoluciones tiene FK directa a cm_facturas_id,
# NO pasa por cm_detalles.

query_distribucion = """
SELECT
    CASE
        WHEN cad_agg.tiene_devolucion IS NOT NULL THEN 'Devuelta (2)'
        WHEN camg_agg.tiene_glosa IS NOT NULL     THEN 'Glosada (1)'
        ELSE                                           'Auditada (0)'
    END AS estado_label,
    COUNT(*)                    AS total_facturas,
    ROUND(AVG(f.valor_factura)) AS valor_promedio_cop,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje
FROM cm_facturas f
LEFT JOIN (
    SELECT cd.cm_facturas_id, 1 AS tiene_glosa
    FROM cm_detalles cd
    INNER JOIN cm_auditoria_motivos_glosas camg ON camg.cm_detalles_id = cd.id
    GROUP BY cd.cm_facturas_id
) camg_agg ON camg_agg.cm_facturas_id = f.id
LEFT JOIN (
    SELECT cad.cm_facturas_id, 1 AS tiene_devolucion
    FROM cm_auditoria_devoluciones cad
    GROUP BY cad.cm_facturas_id
) cad_agg ON cad_agg.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
GROUP BY estado_label
ORDER BY estado_label;
"""

try:
    df_dist = pd.read_sql(query_distribucion, con=engine)
    print("✅ DISTRIBUCIÓN DE CLASES (últimos 24 meses):")
    display(df_dist)
    df_dist.to_csv(OUTPUT_DIR / "01_distribucion_clases.csv", index=False)
    print("✅ Guardado: 01_distribucion_clases.csv")

    estados = df_dist["estado_label"].tolist()
    if "Glosada (1)" in estados and "Devuelta (2)" in estados:
        print("🎯 Las 3 clases están presentes. Modelo multiclase viable.")
    elif "Glosada (1)" in estados:
        print("⚠️  Solo 2 clases (Auditada + Glosada).")
    else:
        print("❌ Solo 1 clase. Revisar joins.")

except Exception as e:
    print(f"❌ Error: {e}")

In [15]:

# Celda 11 — Distribución real de estados: Auditada / Glosada / Devuelta
# CORRECCIÓN (2026-03-04): cm_auditoria_devoluciones tiene FK directa a cm_facturas_id,
# NO pasa por cm_detalles.

query_distribucion = """
SELECT
    CASE
        WHEN cad_agg.tiene_devolucion IS NOT NULL THEN 'Devuelta (2)'
        WHEN camg_agg.tiene_glosa IS NOT NULL     THEN 'Glosada (1)'
        ELSE                                           'Auditada (0)'
    END AS estado_label,
    COUNT(*)                    AS total_facturas,
    ROUND(AVG(f.valor_factura)) AS valor_promedio_cop,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS porcentaje
FROM cm_facturas f
LEFT JOIN (
    SELECT cd.cm_facturas_id, 1 AS tiene_glosa
    FROM cm_detalles cd
    INNER JOIN cm_auditoria_motivos_glosas camg ON camg.cm_detalles_id = cd.id
    GROUP BY cd.cm_facturas_id
) camg_agg ON camg_agg.cm_facturas_id = f.id
LEFT JOIN (
    SELECT cad.cm_facturas_id, 1 AS tiene_devolucion
    FROM cm_auditoria_devoluciones cad
    GROUP BY cad.cm_facturas_id
) cad_agg ON cad_agg.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
GROUP BY estado_label
ORDER BY estado_label;
"""

try:
    df_dist = pd.read_sql(query_distribucion, con=engine)
    print("✅ DISTRIBUCIÓN DE CLASES (últimos 24 meses):")
    display(df_dist)
    df_dist.to_csv(OUTPUT_DIR / "01_distribucion_clases.csv", index=False)
    print("✅ Guardado: 01_distribucion_clases.csv")

    estados = df_dist["estado_label"].tolist()
    if "Glosada (1)" in estados and "Devuelta (2)" in estados:
        print("🎯 Las 3 clases están presentes. Modelo multiclase viable.")
    elif "Glosada (1)" in estados:
        print("⚠️  Solo 2 clases (Auditada + Glosada).")
    else:
        print("❌ Solo 1 clase. Revisar joins.")

except Exception as e:
    print(f"❌ Error: {e}")

✅ DISTRIBUCIÓN DE CLASES (últimos 24 meses):


,estado_label,total_facturas,valor_promedio_cop,porcentaje
0,Auditada (0),3772917,646277.0,83.6
1,Devuelta (2),139724,2033731.0,3.1
2,Glosada (1),598293,6816873.0,13.3


✅ Guardado: 01_distribucion_clases.csv
🎯 Las 3 clases están presentes. Modelo multiclase viable.


In [13]:
# Celda 12 — Resumen final: archivos generados

print("📁 Archivos generados en data/database_analysis/")
print("="*60)
archivos = sorted(OUTPUT_DIR.glob("*.csv"))
for f in archivos:
    size_kb = f.stat().st_size / 1024
    try:
        df_tmp = pd.read_csv(f)
        print(f"  ✅ {f.name:50s} {size_kb:6.1f} KB | {len(df_tmp)} filas × {len(df_tmp.columns)} cols")
    except Exception:
        print(f"  ✅ {f.name:50s} {size_kb:6.1f} KB")

print()
print("PRÓXIMOS PASOS:")
print("  1. Revisar celda 6: columnas candidatas al TARGET en cm_facturas")
print("  2. Revisar celda 10: confirmar que el JOIN funciona")
print("  3. Revisar celda 11: distribución de clases (¿hay Glosadas y Devueltas?)")
print("  4. Si todo OK → ejecutar 02_queries_dataset_entrenamiento.sql")

📁 Archivos generados en data/database_analysis/
  ✅ 00_conteo_registros.csv                               0.3 KB | 9 filas × 2 cols
  ✅ 99_test_join_glosas.csv                               1.0 KB | 20 filas × 6 cols
  ✅ aa_mala_cm_fe_rips_cargas_muestra.csv                 3.6 KB | 2 filas × 63 cols
  ✅ aa_temp_cm_fe_rips_cargas_muestra.csv                 3.2 KB | 2 filas × 63 cols
  ✅ aseg_afiliados_muestra.csv                            6.2 KB | 2 filas × 200 cols
  ✅ aseg_afiliados_ods_muestra.csv                        1.2 KB | 0 filas × 77 cols
  ✅ cm_auditoria_devoluciones_muestra.csv                 2.1 KB | 2 filas × 30 cols
  ✅ cm_auditoria_motivos_glosas_muestra.csv               0.9 KB | 2 filas × 19 cols
  ✅ cm_detalles_muestra.csv                               1.7 KB | 2 filas × 63 cols
  ✅ cm_factura_estados_muestra.csv                        0.3 KB | 2 filas × 7 cols
  ✅ cm_facturas_muestra.csv                               1.8 KB | 2 filas × 68 cols
  ✅ cm_fe_rips_car